In [1]:
# # =========================
# # MDS→クラスタリング 一括実行スクリプト（ward.D2版）
# # 要件: ①OH/FP2データ, ②6指標, ③top3/cumeig(80%), ④old_MDS_日付配下に保存, ⑤CSV出力
# # =========================

# if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }

# # ---- 設定 ----
# # 基準ディレクトリ（必要に応じて調整）
# base_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
# files     <- c("preprocessed_features_OH.csv", "preprocessed_features_FP.csv")

# # ② 指標
# method_list   <- c("silhouette","dunn","gap","ch","db","ptbiserial")

# # ③ 次元のモード
# dim_mode_list <- c("top3", "cumeig")  # top3:寄与率上位3軸, cumeig:累積>=80%

# # ④ 出力ルート（old_MDS + 実行日）
# today_tag <- format(Sys.Date(), "%Y%m%d")
# root_out  <- file.path(base_dir, paste0("STEP2_old_MDS_", today_tag))
# if (!dir.exists(root_out)) dir.create(root_out, recursive = TRUE)

# set.seed(42)

# # ---- ヘルパー ----
# read_numeric_matrix <- function(path){
#   df <- read.delim(path, header = TRUE, sep = ",", row.names = 1, as.is = TRUE, strip.white = FALSE)
#   is_num <- !vapply(df, is.character, logical(1))
#   num    <- df[, is_num, drop = FALSE]
#   return(num)
# }

# compute_mds_from_corr <- function(numData, k_max = 50){
#   NS <- nrow(numData); NF <- ncol(numData)
#   message(sprintf("[Info] rows=%d, cols=%d", NS, NF))
#   nonNApct <- sum(!is.na(as.matrix(numData))) / (NS*NF) * 100
#   message(sprintf("[Info] Non-NA percentage: %.2f%%", nonNApct))

#   corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
#   corData[is.na(corData)] <- 0
#   ddata <- dist(1 - corData)

#   # cmdscale の安全な次元数（高々NF-1）
#   k <- min(k_max, max(1, ncol(corData) - 1))
#   mds <- cmdscale(ddata, k = k, eig = TRUE)

#   # 固有値と寄与率
#   eig_vals <- mds$eig
#   totaleig <- sum(eig_vals)
#   peigen   <- if (totaleig == 0) rep(0, length(eig_vals)) else eig_vals / totaleig

#   contrib <- data.frame(
#     axis = seq_along(eig_vals),
#     eigenvalue = eig_vals,
#     percent = peigen,
#     cum_percent = cumsum(peigen),
#     stringsAsFactors = FALSE
#   )
#   return(list(mds = mds, contrib = contrib))
# }

# select_dims <- function(contrib_df, mode){
#   # MDSの寄与率が正の軸のみを選択対象に
#   pos_idx <- which(contrib_df$percent > 0)
#   if (length(pos_idx) == 0) return(0)
#   if (mode == "top3"){
#     return(min(3, length(pos_idx)))
#   } else { # cumeig >= 0.8
#     thr <- 0.8
#     k <- which(contrib_df$cum_percent[pos_idx] >= thr)[1]
#     if (is.na(k)) k <- length(pos_idx)
#     return(k)
#   }
# }

# safe_nbclust <- function(X, index_name){
#   # ward.D2 階層クラスタを NbClust で実行（距離はユークリッド）
#   # max.nc は規模に応じて抑制
#   res <- tryCatch({
#     NbClust(data = X,
#             diss = NULL,
#             distance = "euclidean",
#             min.nc = 2,
#             max.nc = min(100, nrow(X) - 1),
#             method = "ward.D2",
#             index = index_name)
#   }, error = function(e) e)
#   return(res)
# }

# # ---- メイン ----
# for (f in files){
#   message(sprintf("=== Processing: %s ===", f))
#   in_path <- file.path(base_dir, f)
#   numData <- read_numeric_matrix(in_path)

#   # MDS計算（相関→距離→cmdscale）
#   m <- compute_mds_from_corr(numData)
#   mds <- m$mds
#   contrib <- m$contrib

#   # 出力フォルダ（データセットごと）
#   dataset_stem <- sub("\\.csv$", "", f)
#   out_dir <- file.path(root_out, dataset_stem)
#   if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

#   # ⑤ MDS寄与度CSV
#   contrib_out <- file.path(out_dir, sprintf("%s_mds_contributions_%s.csv", dataset_stem, today_tag))
#   write.csv(contrib, contrib_out, row.names = FALSE)

#   # MDS座標（相関行列の「変数」ごとに1行）
#   coords <- as.matrix(mds$points)
#   var_ids <- rownames(coords)

#   # ⑤ クラスタ割当とクラスタ数サマリを格納
#   label_df <- data.frame(id = var_ids, stringsAsFactors = FALSE)
#   ksum <- data.frame(mode = character(0), index = character(0), k = integer(0),
#                      stringsAsFactors = FALSE)

#   for (dm in dim_mode_list){
#     k_dim <- select_dims(contrib, dm)
#     if (k_dim <= 0){
#       warning(sprintf("No positive-contribution axes for %s (%s). Skipping.", dataset_stem, dm))
#       next
#     }
#     X <- coords[, 1:k_dim, drop = FALSE]

#     for (idx in method_list){
#       message(sprintf(".. %s | %s | dims=%d (ward.D2)", dm, idx, k_dim))
#       res <- safe_nbclust(X, idx)
#       col_name <- paste(dm, idx, sep = "_")

#       if (inherits(res, "error")){
#         warning(sprintf("NbClust failed: %s | %s | %s", dataset_stem, col_name, res$message))
#         label_df[[col_name]] <- NA_integer_
#         ksum <- rbind(ksum, data.frame(mode = dm, index = idx, k = NA_integer_, stringsAsFactors = FALSE))
#         next
#       }

#       # ベスト分割
#       if (!is.null(res$Best.partition)){
#         cl <- res$Best.partition
#         cl <- cl[var_ids]  # 行順を揃える
#         label_df[[col_name]] <- as.integer(cl)
#       } else {
#         label_df[[col_name]] <- NA_integer_
#       }

#       # ベストクラスタ数
#       k_val <- NA_integer_
#       if (!is.null(res$Best.nc)){
#         if (is.list(res$Best.nc) && !is.null(res$Best.nc[["Number_clusters"]])){
#           k_val <- as.integer(res$Best.nc[["Number_clusters"]])
#         } else if (is.matrix(res$Best.nc) && "Number_clusters" %in% rownames(res$Best.nc)){
#           k_val <- as.integer(res$Best.nc["Number_clusters", 1])
#         } else if (is.numeric(res$Best.nc)){
#           k_val <- as.integer(res$Best.nc[1])
#         }
#       }
#       ksum <- rbind(ksum, data.frame(mode = dm, index = idx, k = k_val, stringsAsFactors = FALSE))
#     }
#   }

#   # ⑤ CSV保存（クラスタ割当／クラスタ数サマリ）
#   labels_out <- file.path(out_dir, sprintf("%s_cluster_labels_%s.csv", dataset_stem, today_tag))
#   ksum_out   <- file.path(out_dir, sprintf("%s_cluster_k_summary_%s.csv", dataset_stem, today_tag))
#   write.csv(label_df, labels_out, row.names = FALSE)
#   write.csv(ksum, ksum_out, row.names = FALSE)

#   message(sprintf("[Saved] %s", labels_out))
#   message(sprintf("[Saved] %s", ksum_out))
#   message(sprintf("[Saved] %s", contrib_out))
# }

# message("All done.")


Loading required package: NbClust

=== Processing: preprocessed_features_OH.csv ===

[Info] rows=1046, cols=394

[Info] Non-NA percentage: 98.69%

.. top3 | silhouette | dims=3 (ward.D2)

.. top3 | dunn | dims=3 (ward.D2)

.. top3 | gap | dims=3 (ward.D2)

.. top3 | ch | dims=3 (ward.D2)

.. top3 | db | dims=3 (ward.D2)

.. top3 | ptbiserial | dims=3 (ward.D2)



ERROR: Error in coords[, 1:k_dim, drop = FALSE]: subscript out of bounds


In [3]:
# =========================
# MDS → クラスタリング 一括実行（ward.D2 版・安全化パッチ込み）
# 要件:
# ① 入力: preprocessed_features_OH.csv / preprocessed_features_FP.csv を for ループ処理
# ② 指標: silhouette, dunn, gap, ch, db, ptbiserial
# ③ 次元: top3（寄与度上位3軸）/ cumeig（累積寄与率>=80%まで）
# ④ 出力: データごとに "old_MDS_YYYYMMDD/<dataset>/" フォルダを作成して保存
# ⑤ 出力CSV: クラスタリング結果（割当・kサマリ）と MDS 各軸の寄与度
# =========================

if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }

# ---- 設定 ----
base_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
files     <- c("preprocessed_features_OH.csv", "preprocessed_features_FP.csv")

method_list    <- c("silhouette","dunn","gap","ch","db","ptbiserial")  # ②
dim_mode_list  <- c("top3", "cumeig")                                  # ③

today_tag <- format(Sys.Date(), "%Y%m%d")
root_out  <- file.path(base_dir, paste0("old_MDS_", today_tag))        # ④
if (!dir.exists(root_out)) dir.create(root_out, recursive = TRUE)

set.seed(42)

# ---- ヘルパー関数 ----
read_numeric_matrix <- function(path){
  df <- read.delim(path, header = TRUE, sep = ",", row.names = 1, as.is = TRUE, strip.white = FALSE)
  is_num <- !vapply(df, is.character, logical(1))
  if (!any(is_num)) stop("数値列が見つかりません: ", path)
  num <- df[, is_num, drop = FALSE]
  return(num)
}

compute_mds_from_corr <- function(numData, k_max = 50){
  NS <- nrow(numData); NF <- ncol(numData)
  message(sprintf("[Info] rows=%d, cols=%d", NS, NF))
  nonNApct <- sum(!is.na(as.matrix(numData))) / (NS*NF) * 100
  message(sprintf("[Info] Non-NA percentage: %.2f%%", nonNApct))

  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corData[is.na(corData)] <- 0
  ddata <- dist(1 - corData)

  # cmdscale の出力次元は最大で列数-1
  k <- min(k_max, max(1, ncol(corData) - 1))
  mds <- cmdscale(ddata, k = k, eig = TRUE)

  eig_vals <- mds$eig
    # 累積寄与率の計算に絶対値を使用
    totaleig <- sum(abs(eig_vals))
    peigen   <- if (totaleig == 0) rep(0, length(eig_vals)) else abs(eig_vals) / totaleig

  contrib <- data.frame(
    axis        = seq_along(eig_vals),
    eigenvalue  = eig_vals,
    percent     = peigen,
    cum_percent = cumsum(peigen),
    stringsAsFactors = FALSE
  )
  return(list(mds = mds, contrib = contrib))
}

select_dims <- function(contrib_df, mode, avail_dims){
  # 正の寄与軸のみを対象
  pos_idx <- which(contrib_df$percent > 0)

  # 正の寄与が無い場合はフォールバック（1〜3軸の範囲で可能な最小値）
  if (length(pos_idx) == 0) {
    return(min(3, max(1, avail_dims)))
  }

  if (mode == "top3"){
    k <- min(3, length(pos_idx))
  } else {  # cumeig >= 0.8
    thr <- 0.8
    k <- which(contrib_df$cum_percent[pos_idx] >= thr)[1]
    if (is.na(k)) k <- length(pos_idx)
  }

  # 利用可能次元で上限カット＋下限1
  k <- min(k, avail_dims)
  k <- max(1, k)
  return(k)
}

safe_nbclust <- function(X, index_name){
  # ward.D2 階層クラスタ（距離はユークリッド）
  tryCatch({
    NbClust(data = X,
            diss = NULL,
            distance = "euclidean",
            min.nc = 2,
            max.nc = min(100, nrow(X) - 1),
            method = "ward.D2",
            index  = index_name)
  }, error = function(e) e)
}

# ---- メイン処理 ----
for (f in files){
  message(sprintf("=== Processing: %s ===", f))
  in_path <- file.path(base_dir, f)
  numData <- read_numeric_matrix(in_path)

  # MDS（相関→距離→cmdscale）
  m <- compute_mds_from_corr(numData)
  mds     <- m$mds
  contrib <- m$contrib

  # 出力フォルダ（データセット単位）
  dataset_stem <- sub("\\.csv$", "", f)
  out_dir <- file.path(root_out, dataset_stem)
  if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

  # ⑤ MDS 寄与度の保存
  contrib_out <- file.path(out_dir, sprintf("%s_mds_contributions_%s.csv", dataset_stem, today_tag))
  write.csv(contrib, contrib_out, row.names = FALSE)

  # MDS 座標（相関行列の「変数」ごとに1行）
  coords <- as.matrix(mds$points)

  # k=1 の場合はベクトル化されることがあるので行列に強制
  if (is.null(dim(coords))) {
    coords <- matrix(coords, ncol = 1)
    rownames(coords) <- rownames(mds$points)
    colnames(coords) <- "Dim1"
  }

  var_ids    <- rownames(coords)
  avail_dims <- ncol(coords)
  if (avail_dims < 1) stop("MDSの座標次元が取得できません（ncol(coords) < 1）。")

  # 結果テーブル準備
  label_df <- data.frame(id = var_ids, stringsAsFactors = FALSE)
  ksum     <- data.frame(mode = character(0), index = character(0), k = integer(0),
                         stringsAsFactors = FALSE)

  # 次元モード × 指標 でクラスタリング
  for (dm in dim_mode_list){
    k_dim <- select_dims(contrib, dm, avail_dims)
    k_dim <- min(k_dim, avail_dims)  # 念のため再カット

    X <- coords[, 1:k_dim, drop = FALSE]

    for (idx in method_list){
      message(sprintf(".. %s | %s | dims=%d (ward.D2)", dm, idx, k_dim))
      res <- safe_nbclust(X, idx)
      col_name <- paste(dm, idx, sep = "_")

      if (inherits(res, "error")){
        warning(sprintf("NbClust failed: %s | %s | %s", dataset_stem, col_name, res$message))
        label_df[[col_name]] <- NA_integer_
        ksum <- rbind(ksum, data.frame(mode = dm, index = idx, k = NA_integer_, stringsAsFactors = FALSE))
        next
      }

      # ベスト分割
      if (!is.null(res$Best.partition)){
        cl <- res$Best.partition
        cl <- cl[var_ids]  # 行順合わせ
        label_df[[col_name]] <- as.integer(cl)
      } else {
        label_df[[col_name]] <- NA_integer_
      }

      # ベストクラスタ数
      k_val <- NA_integer_
      if (!is.null(res$Best.nc)){
        if (is.list(res$Best.nc) && !is.null(res$Best.nc[["Number_clusters"]])){
          k_val <- as.integer(res$Best.nc[["Number_clusters"]])
        } else if (is.matrix(res$Best.nc) && "Number_clusters" %in% rownames(res$Best.nc)){
          k_val <- as.integer(res$Best.nc["Number_clusters", 1])
        } else if (is.numeric(res$Best.nc)){
          k_val <- as.integer(res$Best.nc[1])
        }
      }
      ksum <- rbind(ksum, data.frame(mode = dm, index = idx, k = k_val, stringsAsFactors = FALSE))
    }
  }

  # ⑤ CSV 保存（ラベル／kサマリ）
  labels_out <- file.path(out_dir, sprintf("%s_cluster_labels_%s.csv", dataset_stem, today_tag))
  ksum_out   <- file.path(out_dir, sprintf("%s_cluster_k_summary_%s.csv", dataset_stem, today_tag))
  write.csv(label_df, labels_out, row.names = FALSE)
  write.csv(ksum,     ksum_out,   row.names = FALSE)

  message(sprintf("[Saved] %s", labels_out))
  message(sprintf("[Saved] %s", ksum_out))
  message(sprintf("[Saved] %s", contrib_out))
}

message("All done.")


=== Processing: preprocessed_features_OH.csv ===

[Info] rows=1046, cols=394

[Info] Non-NA percentage: 98.69%

.. top3 | silhouette | dims=3 (ward.D2)

.. top3 | dunn | dims=3 (ward.D2)

.. top3 | gap | dims=3 (ward.D2)

